# Modelaje financiero y observabilidad

Se entrena un modelo XGBoost para estimar la puntuación crediticia
registrada a partir de las características financieras del cliente.

Se compara su desempeño con una predicción basada en la mediana.

También se implementa una simulación Monte Carlo para explorar escenarios
de ingresos, gastos y saldo durante los siguientes meses.

La estimación del score no representa una probabilidad de incumplimiento.
Los escenarios financieros dependen de supuestos explícitos y no
constituyen pronósticos validados.

Las ejecuciones, métricas, supuestos, errores y alertas se conservan
en una tabla SQLite.

In [6]:
# Importar librerías

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

import hashlib
import json
import sqlite3

import numpy as np
import pandas as pd
import xgboost

from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# Definir rutas

PROYECTO = Path.cwd()

if PROYECTO.name == "notebooks":
    PROYECTO = PROYECTO.parent

RUTA = (
    PROYECTO
    / "data/processed/gold/customer_financial_summary.parquet"
)

RUTA_MENSUAL = (
    PROYECTO
    / "data/processed/gold/customer_monthly_summary.parquet"
)

BD = PROYECTO / "data/observabilidad.sqlite"


# Cargar información

datos = (
    pd.read_parquet(RUTA)
    .sort_values("id")
    .reset_index(drop=True)
)

if datos["id"].isna().any() or datos["id"].duplicated().any():
    raise ValueError(
        "Se requiere una fila por cliente con ID válido."
    )


# Identificar la versión de los datos utilizados

huella = hashlib.sha256(
    pd.util.hash_pandas_object(
        datos,
        index=False
    ).values.tobytes()
).hexdigest()


# Registrar eventos de modelaje

def registrar(evento, estado, detalle, cliente_id=None):
    fila = pd.DataFrame([{
        "evento_id": uuid4().hex,
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "evento": evento,
        "estado": estado,
        "cliente_id": cliente_id,
        "datos_sha256": huella,
        "detalle_json": json.dumps(
            detalle,
            ensure_ascii=False,
            allow_nan=False
        )
    }])

    with sqlite3.connect(BD) as conexion:
        fila.to_sql(
            "eventos_modelaje",
            conexion,
            if_exists="append",
            index=False
        )


print("Clientes:", len(datos))
print("XGBoost:", xgboost.__version__)

Clientes: 2037
XGBoost: 3.2.0


## Entrenar y evaluar XGBoost

Se separan los clientes en entrenamiento, validación y prueba.

La validación determina cuándo detener el entrenamiento.
El conjunto de prueba se reserva para calcular las métricas finales.

Se excluyen los identificadores y los límites de crédito, porque la
imputación de esos límites utilizó la puntuación crediticia que ahora
queremos estimar.

Los valores ausentes se conservan como información no disponible.

Se calculan MAE, RMSE y R². Si XGBoost no mejora el MAE de la mediana,
se registra una alerta.

In [7]:
# Definir variables predictoras

variables = [
    "yearly_income",
    "per_capita_income",
    "total_debt",
    "num_credit_cards",
    "monthly_income",
    "monthly_expenses",
    "monthly_debt_payments",
    "monthly_savings",
    "monthly_transactions",
    "account_balance",
    "transfers_in",
    "transfers_out",
    "outstanding_due",
    "max_days_past_due",
    "income_mean_3",
    "income_std_3",
    "expenses_mean_3"
]


# Entrenar y registrar resultados

def entrenar():
    inicio = perf_counter()

    try:
        X = datos[variables].astype(float)
        y = datos["credit_score"].astype(float)

        if (
            not np.isfinite(y).all()
            or np.isinf(X.to_numpy()).any()
        ):
            raise ValueError(
                "Hay valores no finitos que requieren revisión."
            )

        # Separar clientes: 60 % entrenamiento,
        # 20 % validación y 20 % prueba

        X_desarrollo, X_test, y_desarrollo, y_test = (
            train_test_split(
                X,
                y,
                test_size=0.20,
                random_state=42
            )
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_desarrollo,
            y_desarrollo,
            test_size=0.25,
            random_state=42
        )

        # Configurar modelo

        parametros = dict(
            objective="reg:squarederror",
            eval_metric="mae",
            n_estimators=500,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=10,
            tree_method="hist",
            early_stopping_rounds=30,
            random_state=42,
            n_jobs=4
        )

        modelo = XGBRegressor(**parametros)

        modelo.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # Crear referencia

        referencia = DummyRegressor(strategy="median")

        referencia.fit(X_train, y_train)

        # Calcular métricas

        filas = []

        for nombre, real, pred in [
            (
                "Mediana prueba",
                y_test,
                referencia.predict(X_test)
            ),
            (
                "XGBoost prueba",
                y_test,
                modelo.predict(X_test)
            ),
            (
                "XGBoost entrenamiento",
                y_train,
                modelo.predict(X_train)
            )
        ]:
            filas.append({
                "modelo": nombre,
                "MAE": float(
                    mean_absolute_error(real, pred)
                ),
                "RMSE": float(
                    np.sqrt(mean_squared_error(real, pred))
                ),
                "R2": float(
                    r2_score(real, pred)
                )
            })

        metricas = pd.DataFrame(filas)

        # Generar alertas

        mejora = filas[1]["MAE"] < filas[0]["MAE"]

        alertas = (
            []
            if mejora
            else ["XGBOOST_NO_SUPERA_MEDIANA"]
        )

        # Guardar información de la ejecución

        detalle = {
            "objetivo": (
                "credit_score registrado; "
                "no probabilidad de impago"
            ),
            "version_xgboost": xgboost.__version__,
            "parametros": parametros,
            "variables": variables,
            "mejor_iteracion": int(modelo.best_iteration),
            "filas_train": len(X_train),
            "filas_validacion": len(X_val),
            "filas_prueba": len(X_test),
            "nulos_por_variable": (
                X.isna().sum().astype(int).to_dict()
            ),
            "metricas": filas,
            "alertas": alertas,
            "duracion_segundos": perf_counter() - inicio
        }

        registrar(
            "entrenamiento",
            "OK" if mejora else "ALERTA",
            detalle
        )

        return modelo, metricas, alertas

    except Exception as error:
        registrar(
            "entrenamiento",
            "ERROR",
            {
                "tipo_error": type(error).__name__,
                "mensaje": str(error),
                "duracion_segundos": perf_counter() - inicio
            }
        )
        raise


modelo, metricas, alertas_modelo = entrenar()

display(metricas.round(3))
print("Alertas:", alertas_modelo)

,modelo,MAE,RMSE,R2
0,Mediana prueba,49.522,65.006,-0.000
1,XGBoost prueba,48.504,63.563,0.044
2,XGBoost entrenamiento,49.025,64.472,0.094


Alertas: []


## Predecir ingresos del siguiente mes

Se utiliza el historial mensual para estimar el ingreso del mes siguiente.

Las variables contienen información disponible al cierre de cada mes.

Los conjuntos de entrenamiento, validación y prueba se separan
por la fecha del ingreso que se intenta predecir.

Se compara XGBoost con utilizar el ingreso del último mes.

In [8]:
# Cargar historial mensual

mensual = pd.read_parquet(RUTA_MENSUAL)

mensual["month"] = pd.to_datetime(
    mensual["month"]
)

mensual = mensual.sort_values(
    ["client_id", "currency", "month"]
).reset_index(drop=True)


# Validar claves

if mensual.duplicated(
    ["client_id", "currency", "month"]
).any():
    raise ValueError(
        "Existen resúmenes mensuales duplicados."
    )

if mensual["currency"].nunique() != 1:
    raise ValueError(
        "Entrena un modelo por moneda."
    )


# Definir variables predictoras

variables_ingreso = [
    "monthly_income",
    "monthly_expenses",
    "monthly_debt_payments",
    "monthly_savings",
    "monthly_transactions",
    "transfers_in",
    "transfers_out",
    "closing_balance",
    "income_lag_1",
    "income_lag_3",
    "income_mean_3",
    "income_std_3",
    "expenses_mean_3",
    "negative_balance_months_3",
    "outstanding_due",
    "max_days_past_due",
    "month_number"
]

mensual["month_number"] = mensual["month"].dt.month

for columna in variables_ingreso:
    mensual[columna] = pd.to_numeric(
        mensual[columna],
        errors="coerce"
    )


# Crear objetivo del siguiente mes

grupos = mensual.groupby(
    ["client_id", "currency"],
    sort=False
)

mensual["target_month"] = grupos["month"].shift(-1)
mensual["next_month_income"] = grupos["monthly_income"].shift(-1)

siguiente_mes = mensual["month"] + pd.offsets.MonthBegin(1)

entrenamiento = mensual.loc[
    (mensual["history_months"] >= 4)
    & mensual["target_month"].eq(siguiente_mes)
    & mensual["next_month_income"].notna()
].copy()


# Separar períodos

periodos = sorted(
    entrenamiento["target_month"].unique()
)

if len(periodos) < 10:
    raise ValueError(
        "Se requieren más períodos para separar "
        "entrenamiento, validación y prueba."
    )

inicio_validacion = periodos[-6]
inicio_prueba = periodos[-3]

train = entrenamiento.loc[
    entrenamiento["target_month"] < inicio_validacion
]

validacion = entrenamiento.loc[
    (entrenamiento["target_month"] >= inicio_validacion)
    & (entrenamiento["target_month"] < inicio_prueba)
]

prueba = entrenamiento.loc[
    entrenamiento["target_month"] >= inicio_prueba
]


# Entrenar modelo

modelo_ingreso = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    tree_method="hist",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=4
)

modelo_ingreso.fit(
    train[variables_ingreso],
    train["next_month_income"],
    eval_set=[
        (
            validacion[variables_ingreso],
            validacion["next_month_income"]
        )
    ],
    verbose=False
)


# Comparar modelos

resultados_ingreso = []

for periodo, tabla in [
    ("Validación", validacion),
    ("Prueba", prueba)
]:
    real = tabla["next_month_income"]

    predicciones = {
        "Último ingreso": tabla["monthly_income"].to_numpy(),
        "XGBoost": np.maximum(
            modelo_ingreso.predict(tabla[variables_ingreso]),
            0
        )
    }

    for nombre, pred in predicciones.items():
        resultados_ingreso.append({
            "periodo": periodo,
            "modelo": nombre,
            "MAE": float(mean_absolute_error(real, pred)),
            "RMSE": float(
                np.sqrt(mean_squared_error(real, pred))
            ),
            "R2": float(r2_score(real, pred))
        })

metricas_ingreso = pd.DataFrame(resultados_ingreso)

display(metricas_ingreso.round(3))


# Seleccionar método utilizando validación

validacion_metricas = metricas_ingreso.loc[
    metricas_ingreso["periodo"] == "Validación"
].set_index("modelo")

usar_xgboost = (
    validacion_metricas.loc["XGBoost", "MAE"]
    < validacion_metricas.loc["Último ingreso", "MAE"]
)


# Registrar resultados

huella_mensual = hashlib.sha256(
    pd.util.hash_pandas_object(
        mensual,
        index=False
    ).values.tobytes()
).hexdigest()

registrar(
    "prediccion_ingresos",
    "OK" if usar_xgboost else "ALERTA",
    {
        "objetivo": "Ingreso del siguiente mes",
        "datos_mensuales_sha256": huella_mensual,
        "variables": variables_ingreso,
        "inicio_validacion": str(inicio_validacion),
        "inicio_prueba": str(inicio_prueba),
        "metricas": resultados_ingreso,
        "metodo_seleccionado": (
            "XGBoost" if usar_xgboost else "Último ingreso"
        )
    }
)


# Preparar último mes disponible

ultimos = (
    mensual
    .sort_values("month")
    .groupby(["client_id", "currency"], as_index=False)
    .tail(1)
    .copy()
)


# Reentrenar para predecir el próximo período

if usar_xgboost:
    parametros_finales = modelo_ingreso.get_params()

    parametros_finales.update({
        "n_estimators": int(modelo_ingreso.best_iteration) + 1,
        "early_stopping_rounds": None
    })

    modelo_ingreso_final = XGBRegressor(
        **parametros_finales
    )

    modelo_ingreso_final.fit(
        entrenamiento[variables_ingreso],
        entrenamiento["next_month_income"]
    )

    ingreso_estimado = np.maximum(
        modelo_ingreso_final.predict(
            ultimos[variables_ingreso]
        ),
        0
    )

else:
    ingreso_estimado = ultimos["monthly_income"].to_numpy()


# Guardar predicciones

predicciones_ingreso = ultimos[
    ["client_id", "currency", "month"]
].copy()

predicciones_ingreso = predicciones_ingreso.rename(
    columns={"month": "feature_month"}
)

predicciones_ingreso["prediction_month"] = (
    predicciones_ingreso["feature_month"]
    + pd.offsets.MonthBegin(1)
)

predicciones_ingreso["predicted_income"] = ingreso_estimado

predicciones_ingreso["method"] = (
    "xgboost" if usar_xgboost else "last_month"
)

predicciones_ingreso.to_parquet(
    PROYECTO / "data/processed/gold/income_predictions.parquet",
    engine="pyarrow",
    coerce_timestamps="us",
    allow_truncated_timestamps=False,
    index=False
)

display(predicciones_ingreso.head())

,periodo,modelo,MAE,RMSE,R2
0,Validación,Último ingreso,400.051,599.383,0.912
1,Validación,XGBoost,369.835,566.292,0.922
2,Prueba,Último ingreso,649.356,898.211,0.835
3,Prueba,XGBoost,401.022,653.816,0.913


,client_id,currency,feature_month,prediction_month,predicted_income,method
959,39,USD,2012-01-01,2012-02-01,3609.808105,xgboost
23663,985,USD,2012-01-01,2012-02-01,5211.752441,xgboost
37151,1547,USD,2012-01-01,2012-02-01,4024.047607,xgboost
10823,450,USD,2012-01-01,2012-02-01,3748.183105,xgboost
22031,917,USD,2012-01-01,2012-02-01,3890.149414,xgboost


## Simular escenarios financieros con Monte Carlo

Se utilizan los últimos doce meses disponibles para obtener
valores de referencia de ingresos, gastos y pagos de deuda.

El saldo inicial corresponde al último cierre disponible.

Se generan escenarios para los siguientes meses manteniendo
los supuestos de distribuciones lognormales e independencia.

No se incluyen transferencias futuras. Los pagos de deuda se
aproximan mediante el promedio histórico, no mediante un calendario
de obligaciones futuras.

In [9]:
# Simular escenarios y registrar resultados

def simular_cliente(
    cliente_id,
    gasto_mensual,
    saldo_inicial,
    pago_deuda_mensual,
    ingreso_mensual=None,
    meses=3,
    escenarios=10000,
    variacion_ingreso=0.10,
    variacion_gasto=0.20,
    umbral_alerta=0.20,
    semilla=42
):
    inicio = perf_counter()
    cliente_id = int(cliente_id)

    try:
        cliente = datos.loc[datos["id"] == cliente_id]

        if len(cliente) != 1:
            raise ValueError("Cliente inexistente.")

        # Preparar ingreso base

        usa_ingreso_anual = ingreso_mensual is None

        ingreso = (
            float(cliente.iloc[0]["yearly_income"]) / 12
            if usa_ingreso_anual
            else float(ingreso_mensual)
        )

        # Validar parámetros

        valores = [
            ingreso,
            gasto_mensual,
            pago_deuda_mensual,
            variacion_ingreso,
            variacion_gasto
        ]

        if not np.isfinite(
            valores + [saldo_inicial, umbral_alerta]
        ).all():
            raise ValueError("Los parámetros deben ser finitos.")

        if min(valores) < 0 or not 0 <= umbral_alerta <= 1:
            raise ValueError(
                "Revisa importes, variaciones y umbral."
            )

        if (
            not isinstance(meses, int)
            or not isinstance(escenarios, int)
            or not 1 <= meses <= 120
            or not 100 <= escenarios <= 100000
        ):
            raise ValueError(
                "Meses: 1–120; escenarios: 100–100000."
            )

        # Generar valores con la media y variación indicadas

        rng = np.random.default_rng(semilla)

        def generar(media, coeficiente_variacion):
            sigma = np.sqrt(
                np.log1p(coeficiente_variacion ** 2)
            )

            return media * rng.lognormal(
                mean=-0.5 * sigma ** 2,
                sigma=sigma,
                size=(escenarios, meses)
            )

        ingresos = generar(ingreso, variacion_ingreso)
        gastos = generar(gasto_mensual, variacion_gasto)

        # Calcular saldo acumulado

        saldos = saldo_inicial + np.cumsum(
            ingresos - gastos - pago_deuda_mensual,
            axis=1
        )

        # Resumir escenarios

        tabla = pd.DataFrame({
            "mes": np.arange(1, meses + 1),
            "ingreso_p10": np.quantile(ingresos, 0.10, axis=0),
            "ingreso_p50": np.quantile(ingresos, 0.50, axis=0),
            "ingreso_p90": np.quantile(ingresos, 0.90, axis=0),
            "saldo_p10": np.quantile(saldos, 0.10, axis=0),
            "saldo_p50": np.quantile(saldos, 0.50, axis=0),
            "saldo_p90": np.quantile(saldos, 0.90, axis=0),
            "prob_saldo_negativo": (saldos < 0).mean(axis=0)
        })

        # Generar alertas

        alertas = [
            "ESCENARIO_CON_SUPUESTOS_NO_CALIBRADOS"
        ]

        if usa_ingreso_anual:
            alertas.append(
                "INGRESO_SUPUESTO_ANUAL_DIVIDIDO_ENTRE_12"
            )

        if (
            tabla["prob_saldo_negativo"].max()
            >= umbral_alerta
        ):
            alertas.append(
                "SALDO_NEGATIVO_SUPERA_UMBRAL_DEL_ESCENARIO"
            )

        # Registrar supuestos y resultados

        detalle = {
            "naturaleza": (
                "Simulación ilustrativa; no pronóstico validado"
            ),
            "supuestos": {
                "ingreso_mensual": ingreso,
                "gasto_mensual": gasto_mensual,
                "saldo_inicial": saldo_inicial,
                "pago_deuda_mensual": pago_deuda_mensual,
                "variacion_ingreso": variacion_ingreso,
                "variacion_gasto": variacion_gasto,
                "meses": meses,
                "escenarios": escenarios,
                "semilla": semilla,
                "umbral_alerta": umbral_alerta,
                "distribucion": (
                    "lognormal; independencia entre meses "
                    "e ingresos/gastos"
                )
            },
            "resultados": tabla.to_dict(orient="records"),
            "alertas": alertas,
            "duracion_segundos": perf_counter() - inicio
        }

        registrar(
            "simulacion",
            "ALERTA",
            detalle,
            cliente_id
        )

        return tabla, alertas

    except Exception as error:
        registrar(
            "simulacion",
            "ERROR",
            {
                "tipo_error": type(error).__name__,
                "mensaje": str(error)
            },
            cliente_id
        )
        raise


# Seleccionar cliente

cliente_id = 0

historial_cliente = (
    mensual.loc[
        mensual["client_id"] == cliente_id
    ]
    .sort_values("month")
    .tail(12)
    .copy()
)

if len(historial_cliente) < 6:
    raise ValueError(
        "Se requieren al menos seis meses de historial."
    )


# Calcular valores de referencia

ingresos = historial_cliente["monthly_income"].astype(float)
gastos = historial_cliente["monthly_expenses"].astype(float)
pagos = historial_cliente["monthly_debt_payments"].astype(float)

ingreso_base = float(ingresos.mean())
gasto_base = float(gastos.mean())
pago_base = float(pagos.mean())

saldo_inicial = float(
    historial_cliente.iloc[-1]["closing_balance"]
)

variacion_ingreso = (
    float(ingresos.std(ddof=1) / ingreso_base)
    if ingreso_base > 0
    else 0.0
)

variacion_gasto = (
    float(gastos.std(ddof=1) / gasto_base)
    if gasto_base > 0
    else 0.0
)


# Ejecutar simulación

simulacion, alertas_simulacion = simular_cliente(
    cliente_id=cliente_id,
    gasto_mensual=gasto_base,
    saldo_inicial=saldo_inicial,
    pago_deuda_mensual=pago_base,
    ingreso_mensual=ingreso_base,
    variacion_ingreso=variacion_ingreso,
    variacion_gasto=variacion_gasto
)

display(simulacion.round(3))
print("Alertas:", alertas_simulacion)

,mes,ingreso_p10,ingreso_p50,ingreso_p90,saldo_p10,saldo_p50,saldo_p90,prob_saldo_negativo
0,1,3439.719,4394.176,5613.033,-23143.114,-21814.170,-20403.982,1.0
1,2,3464.031,4392.900,5613.253,-24904.160,-23004.279,-20972.114,1.0
2,3,3430.777,4385.454,5601.953,-26538.361,-24213.221,-21754.025,1.0


Alertas: ['ESCENARIO_CON_SUPUESTOS_NO_CALIBRADOS', 'SALDO_NEGATIVO_SUPERA_UMBRAL_DEL_ESCENARIO']


## Consultar observabilidad y resultados por cliente

La tabla eventos_modelaje conserva el historial de ejecuciones.

Cada evento incluye fecha, estado, versión de los datos y detalles
de las métricas o simulaciones.

La consulta por cliente recupera su última simulación completada,
incluyendo los supuestos y las alertas necesarios para interpretarla.

Para utilizarla desde el asistente, el identificador debe provenir
de la sesión autenticada.

In [10]:
# Consultar el registro de ejecuciones

with sqlite3.connect(BD) as conexion:
    historial = pd.read_sql_query(
        """
        SELECT
            fecha_utc,
            evento,
            estado,
            cliente_id,
            detalle_json
        FROM eventos_modelaje
        ORDER BY fecha_utc DESC
        LIMIT 10
        """,
        conexion
    )

display(historial)


# La aplicación debe pasar el ID de la sesión autenticada

def consultar_ultima_simulacion(cliente_id_sesion):
    with sqlite3.connect(BD) as conexion:
        fila = conexion.execute(
            """
            SELECT fecha_utc, detalle_json
            FROM eventos_modelaje
            WHERE evento = 'simulacion'
              AND estado != 'ERROR'
              AND cliente_id = ?
            ORDER BY fecha_utc DESC
            LIMIT 1
            """,
            (int(cliente_id_sesion),)
        ).fetchone()

    if fila is None:
        return None

    return {
        "fecha_utc": fila[0],
        **json.loads(fila[1])
    }

,fecha_utc,evento,estado,cliente_id,detalle_json
0,2026-10-29T00:46:49.081902+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
1,2026-10-29T00:46:48.148286+00:00,prediccion_ingresos,OK,None,"{""objetivo"": ""Ingreso del siguiente mes"", ""dat..."
2,2026-10-29T00:46:46.045434+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
3,2026-10-29T00:44:16.792204+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
4,2026-10-29T00:44:16.007047+00:00,prediccion_ingresos,OK,None,"{""objetivo"": ""Ingreso del siguiente mes"", ""dat..."
5,2026-10-29T00:44:14.063035+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
6,2026-10-28T06:38:46.189666+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
7,2026-10-28T06:38:45.390785+00:00,prediccion_ingresos,OK,None,"{""objetivo"": ""Ingreso del siguiente mes"", ""dat..."
8,2026-10-28T06:38:43.613293+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
9,2026-10-22T12:26:53.819205+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
